# Validate Silver Layer Relationships
Validates relationships and referential integrity between the Silver tables.

In [0]:
-- Check if all customer_ids in orders exist in customers
SELECT DISTINCT 
o.customer_id
FROM ecommerce_catalog.silver.silver_orders o
LEFT JOIN ecommerce_catalog.silver.silver_customers c 
ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL 
AND o.customer_id IS NOT NULL;

In [0]:
-- Check if all product_ids in orders exist in products
SELECT DISTINCT 
o.product_id
FROM ecommerce_catalog.silver.silver_orders o
LEFT JOIN ecommerce_catalog.silver.silver_products p
ON o.product_id = p.product_id
WHERE p.product_id IS NULL 
AND o.product_id IS NOT NULL;

In [0]:
-- Find customer_ids in clickstream that are not in orders
SELECT DISTINCT 
    c.customer_id
FROM ecommerce_catalog.silver.silver_clickstream c
LEFT JOIN ecommerce_catalog.silver.silver_orders o 
    ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL 
  AND c.customer_id IS NOT NULL;

In [0]:
-- Find customer_ids in clickstream that are not in customers
SELECT DISTINCT 
    l.customer_id
FROM ecommerce_catalog.silver.silver_clickstream l
LEFT JOIN ecommerce_catalog.silver.silver_customers c 
    ON l.customer_id = c.customer_id
WHERE c.customer_id IS NULL 
  AND l.customer_id IS NOT NULL;

In [0]:
-- Find product_ids in clickstream that do not exist in products
SELECT DISTINCT 
    l.product_id
FROM ecommerce_catalog.silver.silver_clickstream l
LEFT JOIN ecommerce_catalog.silver.silver_products p 
    ON l.product_id = p.product_id
WHERE p.product_id IS NULL 
  AND l.product_id IS NOT NULL;

In [0]:
SELECT order_id, customer_id
FROM ecommerce_catalog.silver.silver_orders
WHERE customer_id IS NULL;

In [0]:
SELECT DISTINCT customer_id
FROM ecommerce_catalog.silver.silver_orders
ORDER BY customer_id;

In [0]:
SELECT DISTINCT customer_id
FROM ecommerce_catalog.silver.silver_clickstream
ORDER BY customer_id;

In [0]:
SELECT DISTINCT event_timestamp
FROM ecommerce_catalog.silver.silver_clickstream
WHERE event_type = "purchase"
ORDER BY event_timestamp OFFSET 12;

In [0]:
SELECT *
FROM ecommerce_catalog.silver.silver_clickstream
WHERE event_timestamp = "2025-02-08T22:26:31.000+00:00"

In [0]:
SELECT *
FROM ecommerce_catalog.silver.silver_orders
WHERE order_date = "2025-02-08" AND customer_id = "C0482"
ORDER BY order_date;

### **Summary of Silver Relationship Validation**
The relationships between the cleaned Silver tables `orders`, `customers`, `products`, and `clickstream` were verified for referential and domain consistency.

### **Results**
- All `customer_id` values present in `orders` can be mapped to a record in `customers`.
- All `product_id` values present in `orders` can be mapped to a record in `products`.
- The `product_id` values present in `clickstream` can be mapped to the corresponding products in `products`.
- Four `customer_id` values were identified in `clickstream` that do not exist in either `customers` or `orders`. These represent a referential inconsistency in the clickstream source and are documented.
- The relationship between `clickstream` and `orders` was evaluated based on `customer_id`, `product_id`, and date.
- Only a single clickstream event could be unambiguously mapped to an existing order. For the remaining investigated events, no matching order exists even at the date level.

### **Implications for the Gold Layer**
The tables `orders`, `customers`, and `products` exhibit consistent relationships and can be joined for business and revenue analytics.

In contrast, a reliable direct link between `clickstream` and `orders` is not feasible with the existing data. Therefore, clickstream sessions will not be mapped to specific orders retroactively or based on uncertain assumptions.

Instead, `clickstream` will be utilized in the Gold layer as an independent source for web analytics, including:
- Conversion funnels
- Session and user behavior
- Cart and checkout abandonment
- Traffic source analysis
- Device analysis

Direct revenue attribution from clickstream sessions to orders will not be performed due to the lack of reliable linkages.